# Day 9 · Run all three: success, cost, latency

**First-principles question:** Is the extra intelligence worth its price and its wait?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

**First, move your code into `lab/`** (copy and paste, 5 minutes): version B from day 6 into
`lab/version_b.py`, the agent loop from day 8 into `lab/agent.py`. Keep the signatures shown at the top of each
file. From now on every notebook imports them.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lab.acme import load_cases, load_results, save_results
from lab.agent import run_agent
from lab.evals import score

cases = load_cases()
by_id = {c["id"]: c for c in cases}

## 1 · Run version C on every case
Cached, so this is slow only the first time.

In [ ]:
c_rows = []
for case in cases:
    answer, trace = run_agent(case)
    c_rows.append(dict(answer, id=case["id"], steps=len(trace),
                       tokens_in=sum(e["tokens_in"] for e in trace),
                       tokens_out=sum(e["tokens_out"] for e in trace),
                       seconds=sum(e["seconds"] for e in trace)))
save_results("C", c_rows)
print(len(c_rows), "cases; average model calls per case:", np.mean([r["steps"] for r in c_rows]).round(1))

## 2 · Percentiles by hand

In [ ]:
def percentile(values, q):
    """The q-th percentile (0 to 100) by the 'nearest rank' method."""
    # TODO: sort the values; rank = ceil(q / 100 * n) (at least 1); return the value at position rank - 1
    ...

times = [r["seconds"] for r in c_rows]
print("median:", percentile(times, 50), "numpy:", np.percentile(times, 50, method="inverted_cdf"))
print("p95:   ", percentile(times, 95), "numpy:", np.percentile(times, 95, method="inverted_cdf"))

## 3 · Prices
Pick one paid model you might use. Look up its price per **million** tokens on the provider's pricing page, and today's US dollar to rupee rate.

In [ ]:
# TODO: fill in from the pricing page (US dollars per million tokens) and the exchange rate
PRICE_IN_USD = None
PRICE_OUT_USD = None
INR_PER_USD = None

def cost_inr(tokens_in, tokens_out):
    return (tokens_in * PRICE_IN_USD + tokens_out * PRICE_OUT_USD) / 1e6 * INR_PER_USD

## 4 · One table, two charts

In [ ]:
def row(version, rows):
    passed = np.array([score(by_id[r["id"]], r)["passed"] for r in rows], dtype=float)
    p, n = passed.mean(), len(passed)
    secs = [r.get("seconds", 0) for r in rows]
    cost_1000 = 1000 * np.mean([cost_inr(r.get("tokens_in", 0), r.get("tokens_out", 0)) for r in rows])
    return {"version": version, "pass rate": p, "± 2 se": 2 * np.sqrt(p * (1 - p) / n),
            "median s": percentile(secs, 50), "p95 s": percentile(secs, 95),
            "tokens per email": np.mean([r.get("tokens_in", 0) + r.get("tokens_out", 0) for r in rows]),
            "₹ per 1,000 emails": cost_1000, "₹ per correct reply": cost_1000 / 1000 / p if p else float("inf")}

table = pd.DataFrame([row("A", load_results("A")), row("B", load_results("B")), row("C", c_rows)]).set_index("version")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for name, r in table.iterrows():
    ax[0].errorbar(r["₹ per 1,000 emails"], r["pass rate"], yerr=r["± 2 se"], fmt="o", capsize=6, label=name)
    ax[1].errorbar(r["p95 s"], r["pass rate"], yerr=r["± 2 se"], fmt="o", capsize=6, label=name)
ax[0].set(xlabel="₹ per 1,000 emails", ylabel="pass rate", title="Quality vs. cost")
ax[1].set(xlabel="p95 seconds", title="Quality vs. waiting time"); ax[0].legend(); plt.show()
table.round(3)

**Which version would you ship today, and what would change your mind?** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert percentile([5, 1, 3, 2, 4], 50) == 3, "percentile(): the median of 1..5 is 3"
assert percentile([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 95) == 10
assert None not in (PRICE_IN_USD, PRICE_OUT_USD, INR_PER_USD), "Step 3: fill in the prices"
assert len(table) == 3 and len(c_rows) == len(cases)
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")